In [ ]:
%matplotlib inline

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy scikit-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("ETH.csv")

https://results.elections.org.za/home/Downloads/ME-Results

i choose these two dataset because they are relevant to the question i took them from a trusted website and also big enough for the model to not be biased

In [ ]:
data=pd.read_csv("ETH (1).csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.dtypes

DATA CLEANING

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df_before_duplicates = df.copy()

print("Before removing duplicates:", df.shape)
print("Rows involved in duplicates:", df.duplicated(keep=False).sum())
print("Rows that will be removed:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print("After removing duplicates:", df.shape)
print("Remaining exact duplicates:", df.duplicated().sum())

In [ ]:
df.columns

In [ ]:
numeric_df = df.select_dtypes(include=['int64', 'float64'])

print(numeric_df.columns)

In [ ]:
for column in numeric_df.columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_limit) |
        (df[column] > upper_limit)
    ]

    print(f"\nColumn: {column}")
    print(f"Number of outliers: {len(outliers)}")

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data_before_duplicates = df.copy()

print("Before removing duplicates:", data.shape)
print("Rows involved in duplicates:", data.duplicated(keep=False).sum())
print("Rows that will be removed:", data.duplicated().sum())

In [ ]:
data= data.drop_duplicates().reset_index(drop=True)

print("After removing duplicates:", data.shape)
print("Remaining exact duplicates:", data.duplicated().sum())

In [ ]:
data.columns

In [ ]:
numeric_data = data.select_dtypes(include=['int64', 'float64'])

print(numeric_data.columns)

In [ ]:
for column in numeric_data.columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = data[
        (df[column] < lower_limit) |
        (df[column] > upper_limit)
    ]

    print(f"\nColumn: {column}")
    print(f"Number of outliers: {len(outliers)}")

EDA

In [ ]:
df.hist(figsize=(10,15))
plt.show()

In [ ]:
data.hist(figsize=(10,15))
plt.show()

In [ ]:

import seaborn as sns

df = pd.read_csv('ETH (1).csv', skipinitialspace=True)
df.columns = [c.replace('\ufeff','').strip() for c in df.columns]
df['PartyName'] = df['PartyName'].str.replace(r'\s+',' ',regex=True).str.strip()
df['DateGenerated'] = pd.to_datetime(df['DateGenerated'], dayfirst=True)

pr   = df[df.BallotType == 'PR'].copy()
ward = df[df.BallotType == 'Ward'].copy()

In [ ]:
party_totals = (pr.groupby('PartyName')['TotalValidVotes']
                  .sum()
                  .sort_values(ascending=False))

plt.figure(figsize=(10, 12))
party_totals.plot(kind='barh', color='steelblue')
plt.xlabel('Total valid votes')
plt.ylabel('Party')
plt.title('Total PR votes per party — eThekwini 2016')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('party_totals_pr.png', dpi=150)
plt.show()

In [ ]:
import numpy as np

df = pd.read_csv('ETH (1).csv', skipinitialspace=True)
df.columns = [c.replace('\ufeff','').strip() for c in df.columns]
df['PartyName'] = df['PartyName'].str.replace(r'\s+',' ',regex=True).str.strip()
df['DateGenerated'] = pd.to_datetime(df['DateGenerated'], dayfirst=True)

pr   = df[df.BallotType == 'PR'].copy()
ward = df[df.BallotType == 'Ward'].copy()

In [ ]:
vd = (pr.groupby('VotingDistrict')
        .agg(reg=('RegisteredVoters','first'),
             valid=('TotalValidVotes','sum'),
             spoilt=('SpoiltVotes','first')))

plt.figure(figsize=(8,6))
plt.scatter(vd['reg'], vd['valid'], alpha=0.3, s=10, color='steelblue')
plt.plot([0, vd.reg.max()], [0, vd.reg.max()],
         color='red', linestyle='--', label='turnout = 100%')
plt.xlabel('Registered voters')
plt.ylabel('Total valid votes')
plt.title('Registered voters vs valid votes per voting district')
plt.legend()
plt.tight_layout()
plt.savefig('scatter_reg_vs_valid.png', dpi=150)
plt.show()

In [ ]:
df = pd.read_csv('ETH (1).csv', skipinitialspace=True)
df.columns = [c.replace('\ufeff','').strip() for c in df.columns]
df['PartyName'] = df['PartyName'].str.replace(r'\s+',' ',regex=True).str.strip()
df['DateGenerated'] = pd.to_datetime(df['DateGenerated'], dayfirst=True)

pr   = df[df.BallotType == 'PR'].copy()
ward = df[df.BallotType == 'Ward'].copy()

In [ ]:
def top_n_plus_other(series, n=7, other_label='Other'):
    series = series.sort_values(ascending=False)
    top = series.head(n)
    other = series.iloc[n:].sum()
    if other > 0:
        top[other_label] = other
    return top

In [ ]:
pr_totals = pr.groupby('PartyName')['TotalValidVotes'].sum()
pr_pie = top_n_plus_other(pr_totals, n=7)

plt.figure(figsize=(8,8))
plt.pie(pr_pie, labels=pr_pie.index, autopct='%1.1f%%',
        startangle=90, counterclock=False,
        colors=plt.cm.tab10.colors)
plt.title('PR vote share by party — eThekwini 2016')
plt.tight_layout()
plt.savefig('pie_pr_share.png', dpi=150)
plt.show()


In [ ]:
df.columns

Feature Enginnering

In [ ]:
df = pd.read_csv('ETH (1).csv', skipinitialspace=True)

In [ ]:
print(df.shape)
print(df.head())
print(df.columns.tolist())

In [ ]:
df.columns = [c.replace('\ufeff', '').strip() for c in df.columns]

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df['PartyName'].nunique())
print(sorted(df['PartyName'].unique())[:10])

In [ ]:
df['DateGenerated'] = pd.to_datetime(df['DateGenerated'], dayfirst=True)

In [ ]:
print(df['DateGenerated'].min(), df['DateGenerated'].max())

In [ ]:
pr   = df[df.BallotType == 'PR'].copy()
ward = df[df.BallotType == 'Ward'].copy()

In [ ]:
print('PR rows:  ', len(pr))
print('Ward rows: ', len(pr))

In [ ]:
ward_reg = (pr.groupby(['Ward', 'VotingDistrict'])
              .agg(reg=('RegisteredVoters', 'first'),
                   valid=('TotalValidVotes', 'sum'),
                   spoilt=('SpoiltVotes', 'first'))
              .groupby('Ward').sum())

In [ ]:
print(ward_reg.head())
print(ward_reg.shape)

In [ ]:
ward_reg['turnout']     = ward_reg['valid'] / ward_reg['reg'] * 100
ward_reg['spoilt_rate'] = ward_reg['spoilt'] / (ward_reg['valid'] + ward_reg['spoilt']) * 100
ward_reg['n_vd']        = pr.groupby('Ward')['VotingDistrict'].nunique()

In [ ]:
print(ward_reg[['turnout','spoilt_rate','n_vd']].describe())

In [ ]:
pr_shares = (pr.groupby(['Ward', 'PartyName'])['TotalValidVotes']
               .sum().unstack(fill_value=0))
pr_shares = pr_shares.div(pr_shares.sum(axis=1), axis=0) * 100
pr_shares.columns = ['PR_' + c for c in pr_shares.columns]

In [ ]:
print(pr_shares.shape)
print(pr_shares.sum(axis=1).head())   # should all be 100.0

In [ ]:

ward_shares = (ward.groupby(['Ward', 'PartyName'])['TotalValidVotes']
                   .sum().unstack(fill_value=0))
ward_shares = ward_shares.div(ward_shares.sum(axis=1), axis=0) * 100
ward_shares.columns = ['Ward_' + c for c in ward_shares.columns]

In [ ]:
print(ward_shares.shape)
print('Ward_INDEPENDENT' in ward_shares.columns)

In [ ]:
features = pd.concat([ward_reg, pr_shares, ward_shares], axis=1)
features = features.fillna(0)

In [ ]:
print(features.shape)
print(features.head())

In [ ]:
keep = [
    'reg', 'valid', 'spoilt', 'turnout', 'spoilt_rate', 'n_vd',
    'PR_AFRICAN NATIONAL CONGRESS',
    'PR_DEMOCRATIC ALLIANCE',
    'PR_ECONOMIC FREEDOM FIGHTERS',
    'PR_INKATHA FREEDOM PARTY',
    'Ward_AFRICAN NATIONAL CONGRESS',
    'Ward_DEMOCRATIC ALLIANCE',
    'Ward_INDEPENDENT'
]

features_small = features[keep].copy()
print(features_small.head())

In [ ]:
pr_only = features_small[[c for c in features_small.columns if c.startswith('PR_')]]

features_small['winner']         = pr_only.idxmax(axis=1).str.replace('PR_', '')
features_small['winner_share']   = pr_only.max(axis=1)
features_small['runnerup_share'] = pr_only.apply(lambda r: r.nlargest(2).iloc[-1], axis=1)
features_small['margin']         = features_small['winner_share'] - features_small['runnerup_share']

In [ ]:
print(features_small['winner'].value_counts())
print(features_small['margin'].describe())

Preparing the data

In [ ]:

df_enc = pd.get_dummies(
    df,
    columns=["Ward", "VotingStationName", "PartyName", "BallotType"],
    drop_first=True
)

feature_cols = [c for c in df_enc.columns
                if c not in ["TotalValidVotes", "Province",
                             "Municipality", "DateGenerated",
                             "VotingDistrict"]]

X = df_enc[feature_cols]
y = df_enc["TotalValidVotes"]

print("X shape:", X.shape)
print("y shape:", y.shape)

Training and Splitting

In [ ]:
df.columns

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("ETH.csv", encoding="utf-8-sig")
df.columns = df.columns.str.strip()


print(df["DateGenerated"].nunique())   

In [ ]:

TARGET = "TotalValidVotes"

# Features — every column you listed that carries information
CAT_FEATURES = ["Province", "Municipality", "Ward",
                "VotingDistrict", "VotingStationName",
                "BallotType", "PartyName"]

NUM_FEATURES = ["RegisteredVoters", "SpoiltVotes"]


model_df = df[CAT_FEATURES + NUM_FEATURES + [TARGET]].copy()

# Drop DateGenerated — it is constant, contributes nothing

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = model_df[CAT_FEATURES + NUM_FEATURES]
y = model_df[TARGET]
groups = model_df["VotingDistrict"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train districts:", X_train["VotingDistrict"].nunique())
print("Test districts :", X_test["VotingDistrict"].nunique())
print("No overlap     :", set(X_train["VotingDistrict"]) &
                          set(X_test["VotingDistrict"]) == set())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42,
    stratify=model_df["PartyName"]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         CAT_FEATURES),
        ("num", StandardScaler(), NUM_FEATURES),
    ]
)

In [ ]:
AT_FEATURES = ["Province", "Municipality", "Ward", "BallotType", "PartyName"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model = Pipeline([
    ("prep", preprocessor),
    ("reg",  RandomForestRegressor(n_estimators=300, random_state=42)),
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R² :", round(r2_score(y_test, pred), 3))

In [ ]:
print(y_train.value_counts())

In [ ]:
print("Test set:")
print(y_test.value_counts())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

In [ ]:
models_tuned = {

    "Logistic Regression": LogisticRegression(
        max_iter=500
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=7,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        max_depth=2,
        min_samples_split=20,
        min_samples_leaf=10,
        learning_rate=0.05,
        random_state=42
    )
}

In [ ]:

def build(clf):
    return ImbPipeline([
        ("prep",  make_preprocessor()),
        ("smote", SMOTE(random_state=42, k_neighbors=2)),
        ("clf",   clf),
    ])

models_tuned = {
    "Logistic Regression": build(LogisticRegression(max_iter=2000, C=1.0)),
    "Decision Tree":       build(DecisionTreeClassifier(max_depth=10, random_state=42)),
    "Random Forest":       build(RandomForestClassifier(n_estimators=400,
                                                        max_depth=None,
                                                        random_state=42,
                                                        n_jobs=-1)),
    "Gradient Boosting":   build(GradientBoostingClassifier(n_estimators=200,
                                                            random_state=42)),
    "KNN":                 build(KNeighborsClassifier(n_neighbors=5)),
    "SVM (RBF)":           build(SVC(kernel="rbf", C=1.0, probability=True,
                                     random_state=42)),
    "Naive Bayes":         build(GaussianNB()),
}

In [ ]:
results = []

for name, model in models_tuned.items():

    print(f"\nTraining {name}...")

    # Train — SMOTE applied inside the pipeline
    model.fit(X_train, y_train)

    print(f"Testing {name}...")

    # Training accuracy
    train_pred = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, train_pred)

    # Testing accuracy
    test_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, test_pred)

    difference = train_accuracy - test_accuracy

    results.append({
        "Model":              name,
        "Training Accuracy":  train_accuracy * 100,
        "Testing Accuracy":   test_accuracy * 100,
        "Difference":         difference * 100,
    })

    print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Testing Accuracy:  {test_accuracy * 100:.2f}%")
    print(f"Difference:        {difference * 100:.2f}%")

In [ ]:
results_df = (pd.DataFrame(results)
                .sort_values("Testing Accuracy", ascending=False)
                .reset_index(drop=True))

results_df["Training Accuracy"] = results_df["Training Accuracy"].round(2)
results_df["Testing Accuracy"]  = results_df["Testing Accuracy"].round(2)
results_df["Difference"]        = results_df["Difference"].round(2)

print("\n===== MODEL COMPARISON =====")
print(results_df.to_string(index=False))

In [ ]:
print("Models in dict:", len(models_tuned))
print("Results collected:", len(results))

In [ ]:
pip install imbalanced-learn

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


random_forest = models_tuned["Random Forest"]


y_pred_rf = random_forest.predict(X_test)


cm = confusion_matrix(y_test, y_pred_rf)

print("Confusion Matrix:")
print(cm)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Potable", "Potable"]
)

disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:

importance = random_forest.feature_importances_


feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importance
})


feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

In [ ]:
print("Features used in training:", list(X_train.columns))
print("Model classes:", list(random_forest.classes_))

In [ ]:
import pandas as pd

# ==========================================
# NEW VOTING DISTRICT SAMPLE
# ==========================================

new_district = pd.DataFrame([[
    "KwaZulu-Natal",         # Province
    "ETH - eThekwini",       # Municipality
    "Ward 59500001",         # Ward
    2000,                    # RegisteredVoters
    40,                      # SpoiltVotes
    0.65,                    # Turnout  (65%)
    0.03,                    # Spoilt_rate (3%)
]], columns=X_train.columns)

# ==========================================
# PREDICTION
# ==========================================

prediction = random_forest.predict(new_district)
predicted_party = prediction[0]

print("Prediction:", predicted_party)

# ==========================================
# GET PROBABILITIES
# ==========================================

probability = random_forest.predict_proba(new_district)[0]
classes = list(random_forest.classes_)

print("\nProbability per party:")
for party, prob in zip(classes, probability):
    print(f"  {party:35s}: {prob * 100:.2f}%")


top_idx = probability.argmax()
print(f"\nMost likely winner: {classes[top_idx]} "
      f"({probability[top_idx] * 100:.2f}%)")

In [ ]:
from sklearn.metrics import roc_auc_score

# Get the trained Random Forest model
random_forest = models_tuned["Random Forest"]


y_prob = random_forest.predict_proba(X_test)[:, 1]

# Calculate ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)

print("Random Forest ROC-AUC:", roc_auc)

In [ ]:
import joblib

# Get the trained Random Forest
random_forest = models_tuned["Random Forest"]

# Store model and feature names
model_data = {
    "model": random_forest,
    "features": X_train.columns.tolist()
}

# Save the Random Forest
joblib.dump(model_data, "random_forest_potability_model.pkl")

print("Random Forest model and feature names saved successfully.")

In [ ]:
import joblib

# ==========================================
# LOAD THE SAVED RANDOM FOREST MODEL
# ==========================================

model_data = joblib.load("random_forest_potability_model.pkl")

# Get the model
loaded_model = model_data["model"]

# Get feature names
feature_names = model_data["features"]

print("Random Forest model loaded successfully.")

print("Features:")
print(feature_names)

Forecasting

In [ ]:
pip install pandas

In [ ]:
import pandas as pd

# Load the file
date = pd.read_csv('ETH (1).csv')

# Keep ONLY the DateGenerated column
date= ddate[['DateGenerated']]

# Save to a new file
ddate.to_csv('ETH_timestamps.csv', index=False)

print("Done. Rows:", len(date))

In [ ]:
date.head()

In [ ]:
date= date.interpolate(method="linear")

print("Missing values after interpolation:")
print(df_daily.isnull().sum())

date.head(10)

In [ ]:
!pip install statsmodels

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

print("Ready for ARIMA modeling")